# subset anndata object to only HVG 

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io
import matplotlib.pyplot as plt
import os
import anndata 
import gc
from scipy.sparse import csr_matrix

seed=42
np.random.seed(seed)


main_path = "/rds/general/user/ap5625/home/"

input_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/T2D_REM/Phase_2/objects"

output_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/Ada/perturbation_modeling"

In [2]:
adata = sc.read_h5ad(input_path + "/Adipocytes_annotated_step2.h5ad")

In [4]:
adata.var_names_make_unique()
adata.obs_names_make_unique()  


In [6]:
adata.X.max()

8.916567

In [7]:
adata.layers["raw"].max()

10267.0

In [8]:
adata.X = adata.layers["raw"].copy()

In [9]:
adata.X.max()

10267.0

## subset only to obese and WL - remove lean

In [10]:
adata = adata[adata.obs["condition"].isin(["baseline", "weightloss"])].copy()

In [13]:
print(adata.obs["condition"])

ATAGACCAGACTAGAT-1_1_0       baseline
GCACATACAATGGCCC-1_1_0       baseline
TGAGCGCTCGCCCAGA-1_1_0       baseline
GTAATCGTCTGCGGAC-1_1_0       baseline
TCTACCGGTTGGGAAC-1_1_0       baseline
                              ...    
TTTGGTTAGCTGTCCG-1_18_1    weightloss
TTTGGTTCAGGTGTGA-1_18_1    weightloss
TTTGGTTGTAGCACAG-1_18_1    weightloss
TTTGTTGCACTAGGTT-1_18_1    weightloss
TTTGTTGGTTGTCATG-1_18_1    weightloss
Name: condition, Length: 148228, dtype: category
Categories (2, object): ['baseline', 'weightloss']


## keep the paired donors - present in both conditions 

### check first how many cells id give up if i remove them 

In [17]:
baseline_donors = set(adata.obs[adata.obs['condition'] == 'baseline']['Donor'].unique())
weight_loss_donors = set(adata.obs[adata.obs['condition'] == 'weightloss']['Donor'].unique())


## first step - map the gene symbols to ensembl IDs (but in this iteration - do not drop genes without symbols 

## idea - preprocess the data so that adata.var_names after preprocessing should equal the token dictionary's gene list exactly
## what can i use as filters?
### token_dict_gftokens_gc95M.pkl - to identify genes absent from GeneFormer's vocabulary 
### gene_median_file - genes that have zero/missing median in normaliation reference 
### genes with zero counts across cells - from my matrix 


In [3]:
import pickle

# Load the Geneformer gene→token mapping - they have it in their code 
with open('/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl', 'rb') as f:
    gf_genes = pickle.load(f)


## and then select the HVGs after this 

In [4]:
adata_hvg = adata[:, adata.var['highly_variable']].copy()

In [5]:
print(adata_hvg)

AnnData object with n_obs × n_vars = 155923 × 1969
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'doublet_score', 'predicted_doublet', 'barcodes', 'condition', 'remission', 't2d_batch', 'n_genes', 'count_keep', 'mt_keep', 'rb_keep', 'doublet_keep', 'study', 'pooled_donors', 'SeqID', 'cellID', 'Donor', 'sample', 'Chemistry', 'pool_barcodes', 'dataset', 'BMI', 'sex', 'mt.percent', 'ribo.percent', 'emont_cell_type', 'emont_cell_type2', 'cell_type_level1', 'cell_type_level2', 'cell_state', 'subUMAP1_v1', 'subUMAP2_v1', 'subUMAP1_v2', 'subUMAP2_v2', 'New_donor', 'new_sample', 'UMAPglobal_x', 'UMAPglobal_y', 'orig_barcodes', 'miranda_barcodes', 'leiden_0.1', 'leiden_0.15', 'leiden_0.25', 'leiden_0

In [6]:
adata_hvg.write(output_path + "/adipocytes_subset_hvg_only.h5ad")